# Pre-class: Monday Morning — The Bar to Beat
**⏱ This pre-class notebook takes approximately 15 minutes.**

---

## Scenario: Monday — Marcus's New Brief

It's the start of week 4 at NorthStar Retail. Friday last week Sarah showed Marcus the L03 logistic regression. He nodded, then said:

> *"This is the first model we own. But we're hitting capacity at 200 calls. Can you squeeze MORE recall without more false positives? Try those tree-based models you mentioned."*

This morning Sarah sits down with two coffees, the same NorthStar churn dataset from L03, and a clear goal: **beat the L03 baseline.**

By Friday she has to show:
1. A trained Random Forest classifier.
2. A trained Gradient Boosting classifier.
3. Tuned versions of both — with cross-validated evidence.
4. A recommendation on which model NorthStar should put into production.

Today is just the setup: **re-train the L03 baseline so we see the bar, then try a single decision tree to see why "one tree isn't enough."**

**By the end of this notebook you will be able to:**
- State the L03 baseline F1 we're trying to beat
- See a single deep decision tree overfit dramatically
- Understand why we need an *ensemble* of trees, not just one

In [6]:
# --- Setup: load our toolbox ---
import numpy as np                     # numpy = calculator for lists of numbers
import pandas as pd                    # pandas = tool for working with tables of data (like Excel)
import matplotlib.pyplot as plt        # matplotlib = drawing charts
import seaborn as sns                  # seaborn = prettier charts, built on matplotlib
import warnings                        # lets us hide distracting technical warnings

# Tools from scikit-learn (the machine-learning library):
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold  # split data + fair testing
from sklearn.compose import ColumnTransformer          # apply different prep steps to different columns
from sklearn.pipeline import Pipeline                  # chain prep + model into one tidy assembly line
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # put numbers on same scale; turn categories into numbers
from sklearn.impute import SimpleImputer               # fill in missing values sensibly
from sklearn.linear_model import LogisticRegression    # last week's baseline model
from sklearn.tree import DecisionTreeClassifier        # this week's new model: the decision tree
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score  # scorecards for models

warnings.filterwarnings("ignore")          # hide harmless warning messages so output stays readable
sns.set_style("whitegrid")                 # charts get a light grid background
plt.rcParams["figure.figsize"] = (11, 4.5) # default chart size (width, height in inches)

print("✅ Libraries loaded — ready to compare baseline vs trees")

✅ Libraries loaded — ready to compare baseline vs trees


## Step 1 — Reload the dataset and the L03 pipeline

We use the SAME data and the SAME train/test split as L03 so the comparison is honest.

In [19]:
# --- Load the churn data and split it, exactly as Sarah did in L03 ---
df = pd.read_csv("data/northstar_churn.csv")   # read the customer table from file
y  = df["churned"]                              # y = the answer we want to predict (did they churn? 1/0)
X  = df.drop(columns=["customer_id", "churned"])# X = the clues (everything except the ID and the answer)

# Split into training data (model learns from this) and test data (kept hidden, used to grade the model)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,  # 20% held out; stratify keeps churn % equal in both halves; random_state makes the split repeatable
)

# List which columns are numbers vs categories — they need different preparation
numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

# Preprocessor = the data-prep recipe (same one from L03):
preprocessor = ColumnTransformer([
    # For number columns: fill gaps with the median, then rescale so no column dominates
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    # For category columns: fill gaps with the most common value, then turn each category into 0/1 columns
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])
print(f"Training set: {len(X_train):,} customers ({y_train.mean():.1%} churn)")
print(f"Test set:     {len(X_test):,} customers ({y_test.mean():.1%} churn)")

Training set: 8,000 customers (12.0% churn)
Test set:     2,000 customers (11.9% churn)


## Step 2 — Re-train the L03 logistic regression baseline

This is the model Sarah presented to Marcus last Friday. We re-fit it on the same training set so we have an apples-to-apples F1 to compare against.

In [20]:
# --- Rebuild the L03 baseline: this is the score every new model must beat ---
logreg_pipe = Pipeline([
    ("prep",  preprocessor),                                      # step 1: prepare the data
    ("model", LogisticRegression(max_iter=1000, random_state=42)),# step 2: the L03 model
])
logreg_pipe.fit(X_train, y_train)   # learn from the training customers

# Predict probabilities, apply the threshold Sarah recommended in L03
y_proba_lr = logreg_pipe.predict_proba(X_test)[:, 1]  # for each test customer: probability of churn (0 to 1)
y_pred_lr  = (y_proba_lr >= 0.25).astype(int)         # flag anyone above 25% risk (lower bar = catch more churners)

# Print the scorecard for the baseline
print("=== L03 baseline (LogisticRegression, threshold = 0.25) ===")
print(f"Accuracy:   {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"Precision:  {precision_score(y_test, y_pred_lr):.3f}")
print(f"Recall:     {recall_score(y_test, y_pred_lr):.3f}")
print(f"F1:         {f1_score(y_test, y_pred_lr):.3f}")
print(f"Flagged:    {int(y_pred_lr.sum())} customers")
print()
print(f"→ This is THE BAR. Every model this week has to beat F1 ≈ {f1_score(y_test, y_pred_lr):.3f}.")

=== L03 baseline (LogisticRegression, threshold = 0.25) ===
Accuracy:   0.840
Precision:  0.306
Recall:     0.268
F1:         0.286
Flagged:    209 customers

→ This is THE BAR. Every model this week has to beat F1 ≈ 0.286.


## Step 3 — One naive decision tree

Now let's try the simplest tree-based model: a single `DecisionTreeClassifier` grown to full depth. Note we still wrap it in the same `Pipeline` — the only change is the final step.

(Trees don't actually need scaling — but we keep it in the Pipeline so the comparison is clean. Trees ignore the scaling step's effect.)

In [21]:
# --- First attempt: a single decision tree, left to grow as deep as it wants ---
tree_pipe = Pipeline([
    ("prep",  preprocessor),                                   # same data prep as before
    ("model", DecisionTreeClassifier(random_state=42)),        # default: grow until pure (memorises the training data)
])
tree_pipe.fit(X_train, y_train)   # let the tree learn — it will keep splitting until every training customer is "explained"

# Tree's predict() uses threshold=0.5 internally on probabilities derived from leaf purity
y_pred_tree_train = tree_pipe.predict(X_train)  # score it on customers it has already SEEN
y_pred_tree_test  = tree_pipe.predict(X_test)   # score it on customers it has NEVER seen — the honest test

# Compare train vs test accuracy — a big gap means the tree memorised instead of learned
print("=== Single decision tree (full depth) ===")
print(f"TRAIN accuracy:  {accuracy_score(y_train, y_pred_tree_train):.3f}")
print(f"TEST  accuracy:  {accuracy_score(y_test,  y_pred_tree_test):.3f}")
print(f"Gap:             {accuracy_score(y_train, y_pred_tree_train) - accuracy_score(y_test, y_pred_tree_test):.3f}")
print()
print(f"TEST F1:    {f1_score(y_test, y_pred_tree_test):.3f}")
print(f"TEST recall:{recall_score(y_test, y_pred_tree_test):.3f}")
print(f"TEST prec:  {precision_score(y_test, y_pred_tree_test):.3f}")

=== Single decision tree (full depth) ===
TRAIN accuracy:  1.000
TEST  accuracy:  0.799
Gap:             0.201

TEST F1:    0.183
TEST recall:0.188
TEST prec:  0.178


### 💡 What you should notice

- **Training accuracy is essentially 1.00.** The tree perfectly classifies the training set — it has memorised every customer.
- **Test accuracy is much lower.** The gap is the variance / overfitting symptom.
- **The F1 may actually be HIGHER than logistic regression** at threshold 0.5 — the tree's leaves are individually predictive. But this is brittle: re-run on a different split and the F1 will swing wildly.

This is the textbook overfitting failure mode. One tree memorises; it doesn't generalise.

**The fix isn't to make the tree shallower.** A shallow tree underfits — it can't capture interactions. The fix is to train MANY trees on slightly different subsets and AVERAGE them. That's bagging. That's Random Forest. That's Tuesday's notebook.

## Step 4 — A first shallow tree (just to show the bias-variance tradeoff)

For comparison, here's what happens if we constrain the tree to `max_depth=3`. It can't overfit — but it can't fit much, either.

In [22]:
# --- Second attempt: a SHALLOW tree — only 3 yes/no questions deep ---
shallow_tree = Pipeline([
    ("prep",  preprocessor),                                            # same data prep
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42)),    # max_depth=3 stops the tree from memorising
])
shallow_tree.fit(X_train, y_train)   # learn again, but with the depth limit on

y_pred_st_train = shallow_tree.predict(X_train)  # score on seen customers
y_pred_st_test  = shallow_tree.predict(X_test)   # score on unseen customers

# This time train and test should be close — but is the score good enough?
print("=== Shallow tree (max_depth=3) ===")
print(f"TRAIN accuracy:  {accuracy_score(y_train, y_pred_st_train):.3f}")
print(f"TEST  accuracy:  {accuracy_score(y_test,  y_pred_st_test):.3f}")
print(f"Gap:             {accuracy_score(y_train, y_pred_st_train) - accuracy_score(y_test, y_pred_st_test):.3f}")
print()
print(f"TEST F1:    {f1_score(y_test, y_pred_st_test):.3f}")
print()
print("→ Train and test agree (no overfitting).")
print("→ But the F1 is lower than the deep tree's TEST F1 — the shallow tree can't capture the patterns.")
print()
print("This is the BIAS-VARIANCE tradeoff in one notebook:")
print("  Deep tree:    low bias, HIGH variance (overfits)")
print("  Shallow tree: HIGH bias, low variance (underfits)")
print("  Random Forest will give us low bias AND low variance — by averaging many deep trees.")

=== Shallow tree (max_depth=3) ===
TRAIN accuracy:  0.880
TEST  accuracy:  0.880
Gap:             -0.000

TEST F1:    0.000

→ Train and test agree (no overfitting).
→ But the F1 is lower than the deep tree's TEST F1 — the shallow tree can't capture the patterns.

This is the BIAS-VARIANCE tradeoff in one notebook:
  Deep tree:    low bias, HIGH variance (overfits)
  Shallow tree: HIGH bias, low variance (underfits)
  Random Forest will give us low bias AND low variance — by averaging many deep trees.


### 🎯 Bias and Variance

Think of a dartboard. You're throwing darts trying to hit the bullseye (the true answer).

---

**Bias** = how far off-centre your throws are *on average*

- High bias: all your darts land in the same wrong spot — consistently off
- This happens when your model is **too simple** to capture the real pattern
- Example: fitting a straight line to data that curves — no matter how much data you give it, it'll always be wrong in the same way
- Also called **underfitting**

---

**Variance** = how spread out your throws are

- High variance: your darts are scattered all over the place — inconsistent
- This happens when your model is **too complex** and memorises the training data, including its noise
- Example: a model that nails training data but falls apart on new data — it learned the noise, not the signal
- Also called **overfitting**

---

**The tradeoff**

| | Low Complexity | High Complexity |
|---|---|---|
| **Bias** | High (misses the pattern) | Low |
| **Variance** | Low | High (chases the noise) |

You want a model in the sweet spot — simple enough to generalise, complex enough to learn the real pattern.

---

> **One-liner to remember:** Bias = wrong on average. Variance = inconsistent across datasets. Good models minimise both.

## ✅ Section Summary

| Model | Train acc | Test acc | Test F1 | Diagnosis |
|---|---|---|---|---|
| **L03 Logistic Regression @ 0.25** | — | ~0.84 | ~0.28 | The bar to beat |
| **Single decision tree (full depth)** | ~1.00 | ~0.79 | varies | OVERFITS — memorises training data |
| **Shallow tree (max_depth=3)** | ~0.88 | ~0.88 | very low | UNDERFITS — too simple |

**Key insight:**
> One tree alone is never the answer. Either it overfits (full depth) or it underfits (shallow). Random Forest fixes this by AVERAGING many deep trees, each trained on slightly different data. The variance washes out; the bias stays low.

**Bring to class:**
1. The L03 baseline F1 (the bar to beat).
2. The train-vs-test gap of the deep tree (the overfitting symptom).
3. One question about how Random Forest changes this picture.

---
**In class → Open `02_decision_tree_to_forest.ipynb` first.** That notebook builds the Random Forest properly.